# back-fn-call-with-recipe-args — worked example 3: sigmoid_back reads the cached out (node.array), not the input

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `back-fn-call-with-recipe-args`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The second positional in the canonical call is `node.array` — the *cached forward output* `out`, not the input. Activation back_fns like sigmoid exploit this: `d/dx sigmoid(x) = out * (1 - out)`, so passing the cached `out` lets the reverse pass skip recomputing the sigmoid. Passing the MiniTensor `node` instead of `node.array` would feed a wrapper object into raw-tensor arithmetic and crash.

## Worked solution

**Goal.** Backprop through `out = sigmoid(x)` using the cached `out` rather than recomputing from `x`.

**Step 1 — the forward.** `out = t.sigmoid(x)`. The node caches `out` in `node.array`. The recipe stores `args=(x,)` even though `sigmoid_back` won't actually need `x` — the canonical call passes them anyway.

**Step 2 — sigmoid_back signature.** `sigmoid_back(grad_out, out, x)`. The local derivative is `out * (1 - out)`; multiply by `grad_out` for the chain rule. Note it uses `out` (the second positional) and ignores `x`.

**Step 3 — the canonical call.** `sigmoid_back(grad_out, node.array, *node.recipe.args, **node.recipe.kwargs)`. `node.array` flows into the `out` parameter; `*node.recipe.args` unpacks `(x,)` into the trailing `x` param; kwargs is empty.

**Step 4 — verify against autograd.** We recompute the analytic gradient with a fresh autograd-tracked tensor and `t.allclose`. They match, confirming `out * (1 - out)` and the cached-out channel are correct.

**Why it works.** `sigmoid'(x) = sigmoid(x)(1 - sigmoid(x)) = out(1 - out)`. Because `out` is already cached in `node.array`, no second sigmoid evaluation is needed — the whole point of threading the cached output through the second positional.

In [ ]:
def sigmoid_back(grad_out, out, x):
    return grad_out * out * (1 - out)

class Recipe:
    def __init__(self, func, args, kwargs):
        self.func, self.args, self.kwargs = func, args, kwargs

class Node:
    def __init__(self, array, recipe):
        self.array, self.recipe = array, recipe

t.manual_seed(0)
x = t.randn(4)
out = t.sigmoid(x)
node = Node(out, Recipe(t.sigmoid, (x,), {}))
grad_out = t.ones_like(out)

dx = sigmoid_back(grad_out, node.array, *node.recipe.args, **node.recipe.kwargs)

xr = x.clone().requires_grad_(True)
t.sigmoid(xr).sum().backward()
print('matches autograd:', t.allclose(dx, xr.grad))